# 旺店通店铺与仓库全量查询

调用 `shop.php` 和 `warehouse_query.php`，自动分页读取启用及停用数据，并将结果保存到 `tests/results/`。凭证从 `examples/wangdian_config.py` 或环境变量读取，不会写入结果文件。

In [1]:
import json
import sys
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('未找到项目根目录 pyproject.toml')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wangdian import WangdianClient
from wangdian_inventory.config import load_settings

settings = load_settings()
if not settings.credentials_configured:
    raise ValueError('请先配置 examples/wangdian_config.py 或 WDT_* 环境变量')

OUTPUT_DIR = PROJECT_ROOT / 'tests' / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_SIZE = 100
REQUESTS_PER_MINUTE = 20

print(f'项目目录: {PROJECT_ROOT}')
print(f'接口环境: {settings.environment}')
print(f'输出目录: {OUTPUT_DIR}')

项目目录: /Users/tmt/wangyewangdian
接口环境: production
输出目录: /Users/tmt/wangyewangdian/tests/results


## 分页与去重函数

In [2]:
def fetch_all(client, endpoint, result_key, *, extra_params=None):
    rows = []
    page_no = 0
    extra_params = dict(extra_params or {})

    while True:
        response = client.call(
            endpoint,
            {**extra_params, 'page_no': page_no, 'page_size': PAGE_SIZE},
        )
        batch = response.get(result_key) or []
        if not isinstance(batch, list):
            raise TypeError(f'{endpoint} 的 {result_key} 不是列表')

        rows.extend(batch)
        total_count = response.get('total_count')
        print(
            f'{endpoint}: page={page_no}, 本页={len(batch)}, '
            f'累计={len(rows)}, total={total_count}'
        )

        if len(batch) < PAGE_SIZE:
            break
        if total_count not in (None, '', -1, '-1') and len(rows) >= int(total_count):
            break
        page_no += 1

    return rows


SENSITIVE_FIELDS = {
    'access_token', 'refresh_token', 'session_key', 'app_secret', 'appsecret',
}


def sanitize_record(row):
    return {key: value for key, value in row.items() if key.lower() not in SENSITIVE_FIELDS}


def fetch_enabled_and_disabled(client, endpoint, result_key, id_key):
    unique = {}
    for disabled in (0, 1):
        rows = fetch_all(
            client, endpoint, result_key, extra_params={'is_disabled': disabled}
        )
        for row in rows:
            item = sanitize_record(row)
            item.setdefault('is_disabled', disabled)
            identity = str(item.get(id_key) or '')
            if not identity:
                raise ValueError(f'{endpoint} 返回了缺少 {id_key} 的记录')
            unique[identity] = item
    return sorted(unique.values(), key=lambda item: str(item.get(id_key, '')))

## 查询全部店铺和仓库

In [3]:
with WangdianClient(
    sid=settings.sid,
    app_key=settings.app_key,
    app_secret=settings.app_secret,
    environment=settings.environment,
    timeout=(10, 60),
    requests_per_minute=REQUESTS_PER_MINUTE,
) as client:
    shops = fetch_enabled_and_disabled(
        client, 'shop', 'shoplist', 'shop_id'
    )
    warehouses = fetch_enabled_and_disabled(
        client, 'warehouse_query', 'warehouses', 'warehouse_id'
    )

print(f'店铺总数: {len(shops)}')
print(f'仓库总数: {len(warehouses)}')

shop: page=0, 本页=66, 累计=66, total=66


shop: page=0, 本页=37, 累计=37, total=37


warehouse_query: page=0, 本页=26, 累计=26, total=26


warehouse_query: page=0, 本页=26, 累计=26, total=26
店铺总数: 103
仓库总数: 52


## 保存 JSON

In [4]:
def save_json(filename, value):
    path = OUTPUT_DIR / filename
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print(f'已保存: {path} ({path.stat().st_size:,} bytes)')
    return path


exported_at = datetime.now().astimezone().isoformat(timespec='seconds')
summary = {
    'environment': settings.environment,
    'exported_at': exported_at,
    'shops': {
        'total': len(shops),
        'enabled': sum(int(item.get('is_disabled', 0)) == 0 for item in shops),
        'disabled': sum(int(item.get('is_disabled', 0)) == 1 for item in shops),
    },
    'warehouses': {
        'total': len(warehouses),
        'enabled': sum(int(item.get('is_disabled', 0)) == 0 for item in warehouses),
        'disabled': sum(int(item.get('is_disabled', 0)) == 1 for item in warehouses),
    },
}

shop_path = save_json('shops.json', shops)
warehouse_path = save_json('warehouses.json', warehouses)
summary_path = save_json('summary.json', summary)
summary

已保存: /Users/tmt/wangyewangdian/tests/results/shops.json (110,762 bytes)
已保存: /Users/tmt/wangyewangdian/tests/results/warehouses.json (22,312 bytes)
已保存: /Users/tmt/wangyewangdian/tests/results/summary.json (231 bytes)


{'environment': 'production',
 'exported_at': '2026-07-31T14:29:23+08:00',
 'shops': {'total': 103, 'enabled': 66, 'disabled': 37},
 'warehouses': {'total': 52, 'enabled': 26, 'disabled': 26}}

## 结果预览

In [5]:
{
    'shops_preview': shops[:3],
    'warehouses_preview': warehouses[:3],
}

{'shops_preview': [{'fenxiaoNickNo': '',
   'shop_id': '1',
   'shop_no': 'C3',
   'platform_id': 1,
   'sub_platform_id': 0,
   'shop_name': 'TMT运动户外装备店',
   'shop_short_name': '',
   'pay_account_id': None,
   'auth_state': 3,
   'push_rds_id': None,
   'pay_auth_state': None,
   'account_id': '14168185',
   'account_nick': 'kkkkb8',
   'auth_time': '2020-08-03 00:45:39',
   'expire_time': '2021-07-11 05:59:59',
   're_expire_time': None,
   'contact': None,
   'country': None,
   'province': None,
   'city': None,
   'district': None,
   'address': None,
   'telno': None,
   'mobile': None,
   'zip': None,
   'email': None,
   'remark': None,
   'website': None,
   'prop1': '',
   'prop2': '',
   'group_id': None,
   'is_disabled': False,
   'delete_flag': False,
   'shop_priority': None,
   'modified': '2026-06-02 15:33:10',
   'created': '2020-07-13 10:49:12',
   'main_shop_id': None,
   'platform_store_id': None,
   'sort': 52,
   'platform_shop_name': 'TMT运动户外装备店',
   'employee_